# Community Notes 陣営反応分析 — Simple 版 (CS授業用)

上から順にセルを実行してください。
「★ 設定」セルのパラメータだけ編集すれば他は変更不要です。

## Simple 版の特徴
- noteId サンプリングで 1 ショット実行（数分〜15 分）
- quality は LLM ラベル学習済みの重みをハードコード
- 回帰は 4 変数: `type_a + type_b + quality + log_ratings_count`

## 仮説判定
- **TypeA のみ有意** → 陣営反応が主因（仮説支持）
- **TypeA・B 両方有意** → 自然拡散も寄与（仮説部分支持）
- **どちらも非有意** → 別要因、またはサンプル不足

## 実行時間の目安
| SAMPLE_FRAC | 所要時間 | 用途 |
|---|---|---|
| 0.05 | 3〜5 分 | 動作確認 |
| 0.30 | 10〜15 分 | 本番 |
| 1.00 | 30〜60 分 | フル（メモリ注意）|

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ サンプリング
# ====================================================
# noteId を frac 割でランダム抽出し、そのノートに関する ratings / history を
# すべて読み込んでパイプラインに流す。
# 頑健性チェック: SEED を 1, 2, 3 と変えて同じ frac で再実行し、
#   β_typeA の符号・有意性がブレないかを見る。

SAMPLE_FRAC = 0.30    # 0.05 で動作確認、0.30 で本番
SEED        = 42

# ====================================================
# ★ Polarity
# ====================================================
# 各評価者の polarity を「最初の N 件の評価」だけで固定する。
# こうしないと後の評価も混ざり、burst 判定で目的変数を見ることになる
# (循環論法)。
# 大きくする → polarity が安定するがその rater の後続評価を使う量が増える
# 小さくする → 循環論法は抑えられるが polarity が雑音で揺れる

POLARITY_FIRST_N = 50

# ====================================================
# ★ Burst 検出
# ====================================================
# 「min_count 件の連続評価が、そのノートの平均速度の multiplier 倍以上速い」
# をバーストと定義する。
# MULTIPLIER を下げる / MIN_COUNT を下げる → より多く検出 (弱いバーストも拾う)
# MULTIPLIER を上げる / MIN_COUNT を上げる → より厳しく (顕著なバーストだけ)

BURST_MULTIPLIER = 3.0    # 平均速度の何倍でバーストか
BURST_MIN_COUNT  = 5      # バーストとみなす最小評価数

# ====================================================
# ★ データ場所 (Drive 上の zip フォルダ)
# ====================================================
DRIVE_DATA = '/content/drive/Shareddrives/基礎プロジェクト/data'

# ====================================================
# ★ 結果の保存先 (MyDrive)
# ====================================================
SAVE_DIR = '/content/drive/MyDrive/toriumi_x3_results_simple'

# ====================================================
# ★ GitHub
# ====================================================
REPO_URL = 'https://github.com/hirototoda/toriumi_x3.git'
REPO_DIR = '/content/toriumi_x3'

---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.exists(DRIVE_DATA):
    print(f'OK: フォルダが見つかりました → {DRIVE_DATA}')
    print()
    print('中身:')
    for d in sorted(os.listdir(DRIVE_DATA)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA} が見つかりません')
    print()
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('利用できる共有ドライブ:')
        for d in os.listdir(sd):
            print(f'  {d}')
    print('MyDrive (上位 20 件):')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]:
        print(f'  {d}')

In [ ]:
# GitHub から clone + pull + 依存インストール
!git clone {REPO_URL} {REPO_DIR} 2>/dev/null || echo 'already cloned'
os.chdir(REPO_DIR)
!git pull
!pip install -q statsmodels

In [ ]:
# Drive 上の zip を Colab ローカル (data/raw/) で解凍。
# 既に TSV が存在すればスキップ (再実行安全)。
#
# Drive 側のサブフォルダ名はスペース有無や大小が表記ゆれしているため、
# 候補リストから先に存在するものを採用する。

import zipfile

raw_dir = f'{REPO_DIR}/data/raw'
os.makedirs(raw_dir, exist_ok=True)

def _unzip_all(src_folder, prefix):
    if not os.path.exists(src_folder):
        print(f'  [skip] {src_folder} が無い')
        return
    zips = sorted(f for f in os.listdir(src_folder) if f.startswith(prefix) and f.endswith('.zip'))
    if not zips:
        print(f'  [skip] {prefix}-*.zip が {src_folder} に無い')
        return
    for f in zips:
        tsv_name = f.replace('.zip', '.tsv')
        if os.path.exists(os.path.join(raw_dir, tsv_name)):
            print(f'  [skip exists] {tsv_name}')
            continue
        print(f'  [unzip] {f}')
        with zipfile.ZipFile(os.path.join(src_folder, f)) as zf:
            zf.extractall(raw_dir)

def _pick_subfolder(candidates):
    for name in candidates:
        full = os.path.join(DRIVE_DATA, name)
        if os.path.exists(full):
            return full, name
    return None, candidates[0]  # 見つからなくても最後にメッセージ用に先頭を返す

# (zip prefix, Drive 上のサブフォルダ候補) — スペース有無の表記ゆれを吸収
TARGETS = [
    ('ratings',           ['RatingData', 'Rating data', 'ratings']),
    ('notes',             ['Notes data', 'NotesData', 'notes']),
    ('noteStatusHistory', ['Note status history data', 'NoteStatusHistoryData', 'noteStatusHistory']),
]

for prefix, candidates in TARGETS:
    folder, used = _pick_subfolder(candidates)
    label = used if folder else f'{candidates[0]} (見つからず)'
    print(f'\n● {label} ({prefix}-*)')
    if folder:
        _unzip_all(folder, prefix)
    else:
        tried = ' / '.join(candidates)
        print(f'  [error] {DRIVE_DATA} 直下に {tried} のいずれも存在しない')

print('\n--- data/raw の中身 ---')
for f in sorted(os.listdir(raw_dir)):
    size_mb = os.path.getsize(os.path.join(raw_dir, f)) / (1024 * 1024)
    print(f'  {f:50s} {size_mb:8.1f} MB')


---
## 1. パイプライン実行

以下を順に実行:
1. notes 全件読 → 政治トピックフィルタ → noteId を `SAMPLE_FRAC` で抽出
2. 該当 ratings / history を読み込み
3. polarity を TruncatedSVD で計算 (`POLARITY_FIRST_N` 件で固定)
4. バースト検出 (`BURST_MULTIPLIER`, `BURST_MIN_COUNT`) + TypeA/B 分類
5. quality を LLM 学習済係数で計算
6. ロジスティック回帰 1 本 (相関行列 + VIF 診断も自動 print)

実行時間: SAMPLE_FRAC=0.30 で 10〜15 分目安。

In [ ]:
import subprocess, sys, os

os.chdir(REPO_DIR)

# subprocess.run + check=False で exit code を Python 側で確認できるようにする。
# `!python ...` だと script が落ちても cell は緑のまま、後続の read_csv が
# FileNotFoundError になって混乱するので避ける。
proc = subprocess.run(
    [sys.executable, 'scripts/run_simple.py',
     '--sample-frac',      str(SAMPLE_FRAC),
     '--seed',             str(SEED),
     '--polarity-first-n', str(POLARITY_FIRST_N),
     '--burst-multiplier', str(BURST_MULTIPLIER),
     '--burst-min-count',  str(BURST_MIN_COUNT)],
    check=False,
)

# 1) script が non-zero で終わっていたらここで止める
if proc.returncode != 0:
    raise RuntimeError(
        f'run_simple.py が exit code {proc.returncode} で失敗しました。'
        ' 上のスクリプト出力 (Traceback など) を確認してください。'
    )

# 2) script が success でも出力ファイルが無ければ止める (二重チェック)
expected = 'data/processed/simple_features.csv'
if not os.path.exists(expected):
    print('[debug] data/processed の中身:')
    if os.path.isdir('data/processed'):
        for f in sorted(os.listdir('data/processed')):
            sz = os.path.getsize(f'data/processed/{f}')
            print(f'  {f}  ({sz:,} bytes)')
    else:
        print('  (data/processed ディレクトリ自体が無い)')
    raise RuntimeError(
        f'{expected} が生成されていません。'
        ' 上のスクリプト出力を確認してください。'
    )

print(f'\nOK: {expected} ({os.path.getsize(expected):,} bytes) 生成完了')


---
## 2. 結果の確認

In [ ]:
# 2-1. 特徴量テーブル (回帰の入力)
import os
import pandas as pd

# セル 0 の os.chdir がカーネル再起動等で効いていない場合の保険
os.chdir(REPO_DIR)

feat = pd.read_csv('data/processed/simple_features.csv')
print(f'features: {len(feat):,} notes')
print(f"  deleted (Not Helpful) = {int(feat['deleted'].sum())}")
print(f"  helpful               = {int((feat['deleted']==0).sum())}")
print(f"  typeA bursts          = {int(feat['type_a'].sum())}")
print(f"  typeB bursts          = {int(feat['type_b'].sum())}")
display(feat.head(10))

In [ ]:
# 2-2. バースト一覧
burst_path = 'data/processed/simple_bursts.csv'
if os.path.exists(burst_path):
    bursts = pd.read_csv(burst_path)
    print(f'bursts: {len(bursts):,}')
    print('\nTypeA/B 分布:')
    print(bursts['burst_type'].value_counts(dropna=False))
    display(bursts.head())
else:
    print(f'{burst_path} が無い (バースト検出 0 件)')

In [ ]:
# 2-3. 回帰 summary (テキスト全文)
# type_a の β と p 値を見て仮説を判定する。
print(open('data/processed/simple_regression.txt').read())

---
## 3. 結果を Drive に保存

In [ ]:
# simple_*.csv と simple_*.txt を SAVE_DIR にコピー
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)
copied = 0
for f in os.listdir('data/processed'):
    if f.startswith('simple_'):
        shutil.copy(os.path.join('data/processed', f), SAVE_DIR)
        print(f'  copied: {f}')
        copied += 1
print(f'\nDone! {copied} ファイルを保存: {SAVE_DIR}')

---
## おまけ: 頑健性チェック

同じ `SAMPLE_FRAC` で `SEED` を 1, 2, 3 と変えて 3 回実行し、β_typeA の符号と有意性がブレないか確認する。

1. 上の「★ 設定」セルで `SEED = 1` に変更
2. セル 1 (パイプライン実行) だけ再実行
3. 同様に `SEED = 2`, `SEED = 3` で実行
4. 3 回の結果で β_typeA が一貫して正 かつ 少なくとも 2 回 p < 0.05 なら頑健

※ 出力ファイルは毎回上書きされるので、比較したいときは手動で save dir を分けてください。